In [ ]:
# Import libraries

import os
import json
import openai
from dotenv import load_dotenv
from neo4j import GraphDatabase

In [ ]:
# Load environment variables
load_dotenv()

# Neo4j connection settings (set these in your .env file)
URI = os.getenv("NEO4J_URI", "bolt://localhost:7687")
USER = os.getenv("NEO4J_USER", "neo4j")
PASSWORD = os.getenv("NEO4J_PASSWORD")
DB_NAME = os.getenv("NEO4J_DB", "hydrologykg")
EMBEDDING_MODEL = "text-embedding-3-large"


if not PASSWORD:
    raise ValueError("Neo4j password is missing. Set it in your .env file.")

driver = GraphDatabase.driver(URI, auth=(USER, PASSWORD))

In [ ]:
def load_paper_kg_to_neo4j(paper_id):
    """Loads entity, relationship, and metadata information for a given paper into Neo4j."""

    paths = {
        "entities": f"Paper_KGs/kg_Entities_{paper_id}.json",
        "relations": f"Paper_KGs/Kg_Relationships_{paper_id}.json",
        "metadata": f"Paper_KGs/kg_Metadata_{paper_id}.json"
    }

    with driver.session(database=DB_NAME) as session:

        # Load metadata entities
        with open(paths["metadata"], encoding="utf-8") as f:
            metadata_kg = json.load(f)

        for node in metadata_kg["Entities"]:
            session.run("""
                MERGE (n:Entity {id: $id})
                SET n += $props
            """, id=node["id"], props=node["properties"] | {"label": node["label"]})

        for rel in metadata_kg["Relationships"]:
            session.run(f"""
                MATCH (a:Entity {{id: $source}})
                MATCH (b:Entity {{id: $target}})
                MERGE (a)-[r:`{rel['type']}`]->(b)
            """, source=rel["source"], target=rel["target"])

        # Load extracted entities
        with open(paths["entities"], encoding="utf-8") as f:
            kg_data = json.load(f)

        extracted_nodes = []
        for group in kg_data["Entities"]:
            for ent in group["result"].get("entities", []):
                extracted_nodes.append({
                    "id": ent["id"],
                    "name": ent["name"],
                    "category": ent["category"],
                    "properties": ent.get("properties", {})
                })

        for node in extracted_nodes:
            session.run("""
                MERGE (n:Entity {id: $id})
                SET n.name = $name,
                    n.category = $category,
                    n += $properties
            """, id=node["id"], name=node["name"], category=node["category"], properties=node["properties"])

        # Create extracted relationships
        with open(paths["relations"], encoding="utf-8") as f:
            rel_data = json.load(f)

        for rel in rel_data["Relationships"]:
            session.run(f"""
                MATCH (a:Entity {{id: $source}})
                MATCH (b:Entity {{id: $target}})
                MERGE (a)-[r:`{rel['type']}`]->(b)
            """, source=rel["source"], target=rel["target"])

        # Link metadata paper node to all extracted nodes
        paper_node_id = f"meta_{paper_id}_001"
        for node in extracted_nodes:
            session.run("""
                MATCH (a:Entity {id: $source})
                MATCH (b:Entity {id: $target})
                MERGE (a)-[:HAS_COMPONENT]->(b)
            """, source=paper_node_id, target=node["id"])

    print(f"✅ KG for paper {paper_id} loaded into Neo4j.")

In [ ]:
def get_openai_embedding(text, model=EMBEDDING_MODEL):
    """Fetch embedding vector for the given text using OpenAI."""
    try:
        response = openai.embeddings.create(input=text, model=model)
        return response.data[0].embedding
    except Exception as e:
        print(f"❌ Error generating embedding: {e}")
        return None

In [ ]:
def create_embeddings_openai():
    """Generates and stores OpenAI embeddings for nodes and relationships in Neo4j."""
    driver = GraphDatabase.driver(URI, auth=AUTH)
    with driver.session(database=DB_NAME) as session:
        # Step 1: Remove previous embeddings
        session.run("MATCH (n) REMOVE n.embedding")
        session.run("MATCH ()-[r]->() REMOVE r.embedding")

        # Step 2: Nodes
        node_query = "MATCH (n:Entity) RETURN n, elementId(n) AS node_id"
        nodes = session.run(node_query)

        for record in nodes:
            props = record["n"]
            node_id = record["node_id"]

            # Build text from all non-embedding properties
            text_repr = " ".join(
                f"{key}: {str(value)}" for key, value in props.items() if key != "embedding"
            )

            embedding = get_openai_embedding(text_repr)
            if embedding:
                session.run(
                    "MATCH (n) WHERE elementId(n) = $node_id SET n.embedding = $embedding",
                    node_id=node_id,
                    embedding=embedding
                )

        # Step 3: Relationships
        rel_query = """
        MATCH (a)-[r]->(b)
        RETURN a.name AS source, type(r) AS rel_type, b.name AS target, r, elementId(r) AS rel_id
        """
        rels = session.run(rel_query)

        for record in rels:
            props = record["r"]
            rel_id = record["rel_id"]

            source = record.get("source", "unknown")
            rel_type = record["rel_type"]
            target = record.get("target", "unknown")

            prop_text = " ".join(
                f"{key}: {str(value)}" for key, value in props.items() if key != "embedding"
            )
            rel_repr = f"{source} {rel_type} {target} {prop_text}"

            embedding = get_openai_embedding(rel_repr)
            if embedding:
                session.run(
                    "MATCH ()-[r]->() WHERE elementId(r) = $rel_id SET r.embedding = $embedding",
                    rel_id=rel_id,
                    embedding=embedding
                )

    driver.close()
    print("✅ Embeddings created and stored in Neo4j using OpenAI model.")

In [ ]:
create_embeddings_openai()